# 블랙프라이데이 심화 분석

**목표**: 블랙프라이데이(BF) 기간 구매자·매출·리텐션·특징을 파악하고, 내년 BF 준비 방향과 마케팅 가설을 정리합니다.

**다루는 주제**  
0. 캠페인 대시보드 지표 (참고)  
1. BF 구매자: 신규 vs 재방문  
2. BF 때 주로 무엇을 구매했는지 (카테고리·상품 TOP)  
3. BF 매출액·객단가·같이 많이 팔린 조합  
4. 시장 검색 관심(Trends) vs Olist 실제 구매  
5. BF 방문자 리텐션  
6. BF 유입 vs 비BF 유입 리텐션 비교  
7. BF 유독 많이 구매한 사람의 특징 (지역, 결제, 단가)  
8. 마케팅·쿠폰 제안 (가설)  
9. 기타 이벤트: 브라질 쇼핑 시즌 & Olist 피크  
10. 결론 및 인사이트  

**데이터**: Olist `data/` 또는 kagglehub. (선택) Google Trends CSV.

---
## 캠페인 성과 대시보드에서 자주 쓰는 지표

이벤트(블랙프라이데이 등) 캠페인 성과를 볼 때 대시보드에 넣어두면 좋은 지표 예시입니다. Olist 데이터로 계산 가능한 것 위주로 정리했습니다.

| 구분 | 지표 | 설명 | 본 노트북 반영 |
|------|------|------|----------------|
| **유입·구성** | 신규 vs 재방문 비율 | 해당 기간 구매자 중 첫 구매(신규) vs 기존 고객(재방문) | §2 |
| **규모** | 주문 수, 매출, 객단가(AOV) | 이벤트 기간 총 주문·매출·주문당 평균 결제액 | §4 |
| **성장** | 전주·전년 대비 증감률 | 이벤트 주 vs 비이벤트 주, 또는 YoY | §1(피크일)·§4 |
| **상품** | 카테고리·상품 TOP N, 매출 비중 | 무엇이 많이 팔렸는지 | §3 |
| **함께 구매** | 같이 많이 산 카테고리/상품 쌍 | 번들·크로스셀 후보 | §4 |
| **리텐션** | 이벤트 유입 코호트의 N개월 재구매율 | 이벤트로 들어온 고객이 얼마나 다시 사는지 | §6, §6-2 |
| **비교** | 이벤트 유입 vs 비이벤트 유입 리텐션 | 이벤트 유입이 더 잘 남는지/덜 남는지 | §6-2 |
| **고객 프로파일** | 지역·결제수단·객단가·카테고리 | 이벤트에 많이 산 사람 특징 | §7 |
| **가격** | 이벤트 기간 vs 평소 카테고리별 평균 단가 | 할인 효과 추정(데이터 한계 내) | §7 |

*실제 쿠폰·마케팅 채널 데이터는 없어 전환·채널별 성과는 제한적입니다.*

---
## 1. 데이터 로드 및 블랙프라이데이 기간 정의 (데이터 기반)

**설명**: Olist **실제 주문·매출 데이터**에서 가장 튀는 날(또는 주)을 찾아 블랙프라이데이를 정의합니다.  
달력상 11월 넷째 주가 아니라, **연도별 11월 중 주문 수가 최대인 날**을 피크로 두고, 그 날이 속한 **주(월요일 시작)**를 BF 주로 씁니다.  
예: 2017-11-24가 전년 대비 가장 튀면 → 2017년 BF 주는 11/24가 포함된 그 주로 설정.

**코드 흐름**: 데이터 로드 → 일별 주문 수·매출 집계 → 연도별 11월에서 피크일 추출 → 해당 주(월요일 기준)를 BF 주로 지정.

In [19]:
import pandas as pd
import numpy as np
from pathlib import Path
from itertools import combinations

# ----- 1) 데이터 경로: data/ 우선, 없으면 kagglehub -----
DATA_DIR = Path.cwd() / "data"
for _ in [Path.cwd() / "data", Path.cwd().parent / "예측 대시보드 용 프로젝트" / "data"]:
    if (_ / "orders_delivered.csv").exists():
        DATA_DIR = _
        break

if not (DATA_DIR / "orders_delivered.csv").exists():
    import kagglehub
    _path = Path(kagglehub.dataset_download("olistbr/brazilian-ecommerce"))
    orders = pd.read_csv(_path / "olist_orders_dataset.csv")
    orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"], errors="coerce")
    orders = orders[orders["order_status"] == "delivered"].copy()
    order_items = pd.read_csv(_path / "olist_order_items_dataset.csv")
    products = pd.read_csv(_path / "olist_products_dataset.csv")
    customers = pd.read_csv(_path / "olist_customers_dataset.csv")
    order_payments = pd.read_csv(_path / "olist_order_payments_dataset.csv")
    products["product_category_name"] = products["product_category_name"].fillna("unknown")
    order_payments["payment_type"] = order_payments["payment_type"].replace("not_defined", "unknown")
    print("(data/ 없음 → kagglehub에서 로드)")
else:
    orders = pd.read_csv(DATA_DIR / "orders_delivered.csv")
    order_items = pd.read_csv(DATA_DIR / "order_items.csv")
    products = pd.read_csv(DATA_DIR / "products.csv")
    customers = pd.read_csv(DATA_DIR / "customers.csv")
    order_payments = pd.read_csv(DATA_DIR / "order_payments.csv")
    orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"], errors="coerce")

# ----- 2) 주문별 결제액, 고객 키 연결 -----
order_value = order_payments.groupby("order_id")["payment_value"].sum().reset_index().rename(columns={"payment_value": "order_value"})
orders = orders.merge(order_value, on="order_id").merge(customers[["customer_id", "customer_unique_id", "customer_state"]], on="customer_id")

# ----- 3) 주 단위 날짜: 월요일 기준 주 시작일 -----
orders["order_week_start"] = orders["order_purchase_timestamp"].dt.to_period("W-MON").dt.start_time
orders["year"] = orders["order_purchase_timestamp"].dt.year
orders["month"] = orders["order_purchase_timestamp"].dt.month

# ----- 4) 블랙프라이데이 주 정의: Olist 데이터에서 11월 일별 주문 수가 최대인 날 → 그 주를 BF 주 -----
orders["order_date"] = orders["order_purchase_timestamp"].dt.date
daily = orders.groupby("order_date").agg(주문수=("order_id", "nunique"), 매출=("order_value", "sum")).reset_index()
daily["year"] = pd.to_datetime(daily["order_date"]).dt.year
daily["month"] = pd.to_datetime(daily["order_date"]).dt.month
# 연도별 11월 중 주문 수가 최대인 날(피크일) 찾기
nov_daily = daily[daily["month"] == 11]
peak_dates = nov_daily.loc[nov_daily.groupby("year")["주문수"].idxmax()][["year", "order_date", "주문수", "매출"]].sort_values("year").reset_index(drop=True)
peak_dates = peak_dates.rename(columns={"order_date": "peak_date", "주문수": "peak_주문수", "매출": "peak_매출"})
# 피크일이 속한 주의 월요일 = BF 주 시작일 (예: 2017-11-24 → 2017-11-21 주)
bf_weeks = []
for _, row in peak_dates.iterrows():
    week_start = pd.Timestamp(row["peak_date"]).to_period("W-MON").start_time
    bf_weeks.append(week_start)
bf_weeks = sorted(set(bf_weeks))
orders["is_bf_week"] = orders["order_week_start"].isin(bf_weeks)
orders["is_november"] = orders["month"] == 11

print("[데이터 기반] 연도별 11월 피크일 (주문 수 최대인 날) → 해당 주를 BF 주로 사용")
print(peak_dates.to_string(index=False))
print("\nBF 주(월요일 기준):", bf_weeks)

# ----- 5) 주문-상품-카테고리 테이블 (가격은 order_items.price) -----
products["product_category_name"] = products["product_category_name"].fillna("unknown")
ord = order_items.merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
ord["product_category_name"] = ord["product_category_name"].fillna("unknown")
ord = ord.merge(orders[["order_id", "order_purchase_timestamp", "customer_unique_id", "customer_state", "order_week_start", "order_value", "is_bf_week", "is_november", "year"]], on="order_id")

bf_orders = orders[orders["is_bf_week"]]["order_id"].nunique()
nov_orders = orders[orders["is_november"]]["order_id"].nunique()
print("\n[수치] BF 해당 주 주문 건수:", bf_orders)
print("[수치] 11월 전체 주문 건수:", nov_orders)

[데이터 기반] 연도별 11월 피크일 (주문 수 최대인 날) → 해당 주를 BF 주로 사용
 year  peak_date  peak_주문수    peak_매출
 2017 2017-11-24      1147 175250.940

BF 주(월요일 기준): [Timestamp('2017-11-21 00:00:00')]

[수치] BF 해당 주 주문 건수: 3091
[수치] 11월 전체 주문 건수: 7289


---
## 2. BF 구매자: 신규 vs 재방문

**설명**: 블랙프라이데이 해당 주에 구매한 고객이 **그 주가 인생 첫 주문(신규)**인지, **이미 이전에 주문한 적 있는 고객(재방문)**인지 구분합니다.  
고객별 "첫 주문일"을 구한 뒤, BF 주에 낸 주문의 주문일이 해당 고객의 첫 주문일과 같으면 **신규**, 아니면 **재방문**으로 냅니다.

**코드 흐름**: 고객별 첫 주문 시점 집계 → BF 주 주문에 해당 고객의 "첫 주문 여부" 병합 → 신규/재방문 비율·매출 집계.

In [20]:
# ----- 고객별 첫 주문 시점 -----
first_order = orders.groupby("customer_unique_id")["order_purchase_timestamp"].min().reset_index().rename(columns={"order_purchase_timestamp": "first_order_ts"})
orders_with_first = orders.merge(first_order, on="customer_unique_id")
# BF 주에 주문이 있으면 BF 주 기준, 없으면 11월 전체 기준으로 신규/재방문 집계
use_bf_week = orders_with_first["is_bf_week"].any()
event_mask = orders_with_first["is_bf_week"] if use_bf_week else orders_with_first["is_november"]
bf_ord = orders_with_first[event_mask].copy()
bf_ord["is_first_order"] = bf_ord["order_purchase_timestamp"] <= bf_ord["first_order_ts"] + pd.Timedelta(seconds=1)
bf_ord["buyer_type"] = bf_ord["is_first_order"].map({True: "신규", False: "재방문"})

# ----- 신규 vs 재방문: 고객 수, 주문 수, 매출 -----
bf_new_return = bf_ord.groupby("buyer_type").agg(
    고객수=("customer_unique_id", "nunique"),
    주문수=("order_id", "nunique"),
    매출=("order_value", "sum"),
).reset_index()
total_cust = bf_new_return["고객수"].sum()
total_rev_bf = bf_new_return["매출"].sum()
bf_new_return["고객비율_%"] = (bf_new_return["고객수"] / (total_cust or 1) * 100).round(1)
bf_new_return["매출비율_%"] = (bf_new_return["매출"] / (total_rev_bf + 1) * 100).round(1)

event_label_nr = "BF주" if use_bf_week else "11월"
print("[수치]", event_label_nr, "구매자: 신규 vs 재방문")
print(bf_new_return.to_string(index=False))
if len(bf_ord) > 0 and "신규" in bf_new_return["buyer_type"].values:
    print("\n신규 비율(고객 기준):", bf_new_return[bf_new_return["buyer_type"]=="신규"]["고객비율_%"].values[0], "%")

[수치] BF주 구매자: 신규 vs 재방문
buyer_type  고객수  주문수         매출  고객비율_%  매출비율_%
        신규 2998 3023 463692.420  97.800  98.400
       재방문   68   68   7332.990   2.200   1.600

신규 비율(고객 기준): 97.8 %


---
## 3. BF 때 주로 무엇을 구매했는지

**설명**: BF 해당 주(또는 11월)에 **카테고리별** 매출·주문 수, **상품( product_id )별** 판매 건수 TOP N을 냅니다.  
연도별로 나누면 매년 BF 때 인기 카테고리가 어떻게 바뀌는지 볼 수 있습니다.

**코드 흐름**: ord에서 is_bf_week 또는 is_november 필터 → 카테고리별 집계 → 상품별 집계 후 상위 N개.

In [21]:
# ----- 이벤트 기간: BF 주에 주문이 있으면 BF 주, 없으면 11월로 대체 -----
event_ord = ord[ord["is_bf_week"]].copy() if ord["is_bf_week"].any() else ord[ord["is_november"]].copy()
event_label = "BF주" if ord["is_bf_week"].any() else "11월"

# ----- 카테고리별 매출·주문 수 -----
cat_bf = event_ord.groupby("product_category_name").agg(
    매출=("price", "sum"),
    주문수=("order_id", "nunique"),
    상품건수=("order_id", "count"),
).reset_index().sort_values("매출", ascending=False)
cat_bf["매출비중_%"] = (cat_bf["매출"] / (cat_bf["매출"].sum() + 1) * 100).round(1)

print(f"[수치] {event_label} 카테고리별 매출·주문 (상위 15)")
print(cat_bf.head(15).to_string(index=False))

# ----- 상품( product_id )별 판매 건수 TOP 20 -----
product_bf = event_ord.groupby("product_id").agg(
    판매건수=("order_id", "count"),
    매출=("price", "sum"),
    카테고리=("product_category_name", "first"),
).reset_index().sort_values("판매건수", ascending=False)
print(f"\n[수치] {event_label} 상품별 판매 건수 TOP 20")
print(product_bf.head(20).to_string(index=False))

[수치] BF주 카테고리별 매출·주문 (상위 15)
    product_category_name        매출  주문수  상품건수  매출비중_%
          cama_mesa_banho 41693.800  384   465  10.400
       relogios_presentes 35846.730  145   160   8.900
   informatica_acessorios 34247.870  181   220   8.500
             beleza_saude 33987.630  231   251   8.500
         moveis_decoracao 28982.420  242   375   7.200
               brinquedos 24853.590  197   222   6.200
            esporte_lazer 22262.730  207   230   5.500
       ferramentas_jardim 18517.550  238   293   4.600
               cool_stuff 16390.290   93    98   4.100
               perfumaria 16187.670  132   142   4.000
    utilidades_domesticas 13896.160  138   167   3.500
               automotivo 13872.050  100   108   3.500
                telefonia 12188.960  155   168   3.000
                    bebes  8229.910   66    68   2.000
agro_industria_e_comercio  7559.600    5    10   1.900

[수치] BF주 상품별 판매 건수 TOP 20
                      product_id  판매건수       매출               카테

---
## 4. BF 매출액·객단가·같이 많이 팔린 조합

**설명**: BF(또는 11월) 기간 **총 매출**, **객단가(AOV)**, 그리고 **같은 주문 안에서 함께 구매된 카테고리 쌍** 빈도를 계산합니다.  
같이 많이 팔린 조합은 번들·크로스셀 추천 후보로 활용할 수 있습니다.

**코드 흐름**: event_ord로 총매출·주문수·AOV 계산 → 주문별 카테고리 목록 추출 → 2조합 빈도 집계.

In [22]:
# ----- BF(또는 11월) 총 매출·주문 수·객단가 -----
bf_revenue = event_ord.groupby("order_id")["price"].sum().sum()
bf_order_count = event_ord["order_id"].nunique()
bf_aov = bf_revenue / bf_order_count if bf_order_count > 0 else 0
print(f"[수치] {event_label} 총 매출: R$ {bf_revenue:,.0f}")
print(f"[수치] {event_label} 주문 수: {bf_order_count:,}건")
print(f"[수치] {event_label} 객단가(AOV): R$ {bf_aov:,.2f}")

# ----- 같은 주문 내 카테고리 쌍 빈도 (08-F와 동일 로직) -----
order_cats = event_ord.groupby("order_id")["product_category_name"].apply(lambda x: list(x.unique())).reset_index()
pair_count = {}
for cats in order_cats["product_category_name"]:
    if len(cats) < 2:
        continue
    for a, b in combinations(sorted(cats), 2):
        if a != b:
            key = (a, b)
            pair_count[key] = pair_count.get(key, 0) + 1
pair_df = pd.DataFrame([{"cat_a": k[0], "cat_b": k[1], "count": v} for k, v in pair_count.items()]).sort_values("count", ascending=False)
print(f"\n[수치] {event_label} 함께 많이 구매된 카테고리 쌍 TOP 15")
print(pair_df.head(15).to_string(index=False))

[수치] BF주 총 매출: R$ 401,738
[수치] BF주 주문 수: 3,091건
[수치] BF주 객단가(AOV): R$ 129.97

[수치] BF주 함께 많이 구매된 카테고리 쌍 TOP 15
                 cat_a                  cat_b  count
       cama_mesa_banho          casa_conforto      5
       cama_mesa_banho       moveis_decoracao      3
    ferramentas_jardim  utilidades_domesticas      2
            cool_stuff              telefonia      1
          beleza_saude        cama_mesa_banho      1
                 bebes             brinquedos      1
    ferramentas_jardim informatica_acessorios      1
    ferramentas_jardim     relogios_presentes      1
           eletronicos informatica_acessorios      1
          beleza_saude                unknown      1
         esporte_lazer  utilidades_domesticas      1
informatica_acessorios              telefonia      1
                 bebes             cool_stuff      1
            brinquedos              papelaria      1
               unknown  utilidades_domesticas      1


---
## 5. 시장 검색 관심(Trends) vs Olist 실제 구매

**설명**: Google Trends CSV가 있으면 BF 주 vs 비BF 주 **평균 검색 관심도(interest)**를 비교하고,  
Olist BF 주 **카테고리별 매출**과 키워드(카테고리)를 매칭해 "검색 관심 상승이 실제 구매로 이어졌는지" 순위로 비교합니다.

**코드 흐름**: Trends CSV 로드 → BF 주 플래그 → 키워드별 BF vs 비BF 평균 interest → Olist 카테고리와 매칭해 한 테이블로 출력.

In [23]:
TRENDS_CSV = Path.cwd() / "google_trends_br_2016_2018_olist.csv"
for _ in [Path.cwd(), Path.cwd().parent / "예측 대시보드 용 프로젝트"]:
    if (_ / "google_trends_br_2016_2018_olist.csv").exists():
        TRENDS_CSV = _ / "google_trends_br_2016_2018_olist.csv"
        break

if TRENDS_CSV.exists():
    trends = pd.read_csv(TRENDS_CSV)
    trends["date"] = pd.to_datetime(trends["date"], errors="coerce")
    trends["week_start"] = trends["date"].dt.to_period("W-MON").dt.start_time
    trends["is_bf_week"] = trends["week_start"].isin(bf_weeks)
    trends_bf = trends[trends["is_bf_week"]].groupby("keyword")["interest"].mean().reset_index().rename(columns={"interest": "interest_bf"})
    trends_non = trends[~trends["is_bf_week"]].groupby("keyword")["interest"].mean().reset_index().rename(columns={"interest": "interest_non"})
    trends_compare = trends_bf.merge(trends_non, on="keyword", how="inner")
    trends_compare["bf_lift"] = (trends_compare["interest_bf"] - trends_compare["interest_non"]).round(1)
    trends_compare = trends_compare.sort_values("bf_lift", ascending=False)
    print("[수치] Trends: BF 주 vs 비BF 주 평균 검색 관심(interest) 상승 TOP 10")
    print(trends_compare.head(10).to_string(index=False))
    # Olist 카테고리(언더스코어)와 키워드(공백) 매칭
    cat_rev_bf = event_ord.groupby("product_category_name")["price"].sum().reset_index().rename(columns={"price": "olist_bf_revenue"})
    cat_rev_bf["keyword"] = cat_rev_bf["product_category_name"].str.replace("_", " ")
    match = trends_compare.merge(cat_rev_bf, on="keyword", how="inner").sort_values("bf_lift", ascending=False)
    print("\n[수치] 검색 관심 상승(Trends) vs Olist BF 매출 매칭 (일부)")
    print(match.head(10).to_string(index=False))
else:
    print("Google Trends CSV 없음. 02번 노트북 실행 후 google_trends_br_2016_2018_olist.csv 생성 시 위 분석 가능.")

[수치] Trends: BF 주 vs 비BF 주 평균 검색 관심(interest) 상승 TOP 10
         keyword  interest_bf  interest_non  bf_lift
artigos de natal       85.000        16.701   68.300
        pc gamer       68.000        57.064   10.900
      brinquedos       58.000        47.248   10.800
     moveis sala       84.000        73.707   10.300
       papelaria       56.000        45.752   10.200
 cama mesa banho       71.000        63.669    7.300
         bebidas       67.000        60.350    6.600
      automotivo       85.000        78.726    6.300
     eletronicos       33.000        28.777    4.200
      perfumaria       12.000        10.439    1.600

[수치] 검색 관심 상승(Trends) vs Olist BF 매출 매칭 (일부)
              keyword  interest_bf  interest_non  bf_lift product_category_name  olist_bf_revenue
     artigos de natal       85.000        16.701   68.300      artigos_de_natal           382.290
           brinquedos       58.000        47.248   10.800            brinquedos         24853.590
          moveis sala

---
## 6. BF 방문자 리텐션

**설명**: "BF 주에 (최소 1건) 구매한 고객"을 **BF 코호트**로 두고, 그 고객들이 **이후 3개월·6개월 안에 재구매**한 비율을 계산합니다.  
비교용으로 **비BF 주에 첫 구매한 고객**의 동일 기간 재구매율도 구하면, BF 유입 고객이 더 잘 남는지/덜 남는지 해석할 수 있습니다.

**코드 흐름**: BF 주 구매 고객 목록 → 각 고객의 "BF 주 이후 최초 주문일" 또는 "없음" 판단 → N일 내 재구매 여부 집계.

In [24]:
# ----- BF 주에 구매한 고객 목록 (없으면 11월 구매 고객으로 대체) -----
if orders["is_bf_week"].any():
    cohort_orders = orders[orders["is_bf_week"]]
    cohort_label = "BF 주"
else:
    cohort_orders = orders[orders["is_november"]]
    cohort_label = "11월"
bf_buyers = cohort_orders.groupby("customer_unique_id").agg(
    bf_last_order=("order_purchase_timestamp", "max"),
).reset_index()
# 전체 주문에서 코호트 "이후" 주문만 남김
orders_after = orders.merge(bf_buyers[["customer_unique_id", "bf_last_order"]], on="customer_unique_id", how="inner")
orders_after = orders_after[orders_after["order_purchase_timestamp"] > orders_after["bf_last_order"]]
# 각 코호트 구매자별 "이후 첫 재구매일"
first_repurchase = orders_after.groupby("customer_unique_id")["order_purchase_timestamp"].min().reset_index().rename(columns={"order_purchase_timestamp": "first_repurchase_ts"})
bf_buyers = bf_buyers.merge(first_repurchase, on="customer_unique_id", how="left")
bf_buyers["days_to_repurchase"] = (bf_buyers["first_repurchase_ts"] - bf_buyers["bf_last_order"]).dt.days

n_bf = len(bf_buyers)
repurchase_90 = (bf_buyers["days_to_repurchase"] <= 90).sum()
repurchase_180 = (bf_buyers["days_to_repurchase"] <= 180).sum()
print("[수치]", cohort_label, "구매 고객 수:", n_bf)
print("[수치]", cohort_label, "이후 3개월(90일) 내 재구매 고객 수:", repurchase_90, f"→ 리텐션(3개월): {repurchase_90/max(n_bf,1)*100:.1f}%")
print("[수치]", cohort_label, "이후 6개월(180일) 내 재구매 고객 수:", repurchase_180, f"→ 리텐션(6개월): {repurchase_180/max(n_bf,1)*100:.1f}%")

[수치] BF 주 구매 고객 수: 3050
[수치] BF 주 이후 3개월(90일) 내 재구매 고객 수: 37 → 리텐션(3개월): 1.2%
[수치] BF 주 이후 6개월(180일) 내 재구매 고객 수: 52 → 리텐션(6개월): 1.7%


### 6-2. BF 유입 vs 비BF 유입 리텐션 비교

**설명**: "첫 구매가 BF 주인 고객"과 "첫 구매가 BF 주가 아닌 다른 시기인 고객"을 나누어, **동일한 관찰 기간(3개월·6개월)** 안에 재구매한 비율을 비교합니다.  
이벤트로 유입된 고객이 평소 유입 고객보다 더 잘 남는지(리텐션 높음) 아니면 더 빨리 이탈하는지(리텐션 낮음) 해석할 수 있습니다.

**코드 흐름**: 고객별 첫 주문 주가 BF 주인지 여부 → BF 코호트 / 비BF 코호트 각각 90일·180일 내 재구매 여부 집계 → 비율 비교 테이블 출력.

In [25]:
# ----- 고객별 첫 주문 시점 + 그 주가 BF 주인지 여부 -----
first_ord = orders.groupby("customer_unique_id").agg(
    first_ts=("order_purchase_timestamp", "min"),
    order_id_first=("order_id", "first"),
).reset_index()
first_ord["first_week_start"] = first_ord["first_ts"].dt.to_period("W-MON").dt.start_time
first_ord["cohort_type"] = first_ord["first_week_start"].isin(bf_weeks).map({True: "BF 유입", False: "비BF 유입"})

# ----- 관찰 한계: 데이터 상 마지막 주문일 기준으로 90일/180일 관찰 가능한 코호트만 -----
obs_end = orders["order_purchase_timestamp"].max()
cut_90 = obs_end - pd.Timedelta(days=90)
cut_180 = obs_end - pd.Timedelta(days=180)
first_ord["obs_90"] = first_ord["first_ts"] <= cut_90
first_ord["obs_180"] = first_ord["first_ts"] <= cut_180

# ----- 각 고객의 "첫 주문 후 90일/180일 안에 재구매 여부" -----
orders_after_first = orders.merge(first_ord[["customer_unique_id", "first_ts"]], on="customer_unique_id")
orders_after_first = orders_after_first[orders_after_first["order_purchase_timestamp"] > orders_after_first["first_ts"]]
orders_after_first["days_later"] = (orders_after_first["order_purchase_timestamp"] - orders_after_first["first_ts"]).dt.days
repurchase_90 = orders_after_first[orders_after_first["days_later"] <= 90].groupby("customer_unique_id").size().reindex(first_ord["customer_unique_id"]).fillna(0) > 0
repurchase_180 = orders_after_first[orders_after_first["days_later"] <= 180].groupby("customer_unique_id").size().reindex(first_ord["customer_unique_id"]).fillna(0) > 0
first_ord["ret_90"] = first_ord["customer_unique_id"].map(repurchase_90)
first_ord["ret_180"] = first_ord["customer_unique_id"].map(repurchase_180)

# ----- 코호트별 리텐션 (관찰 가능한 고객만) -----
def retention_table(df, obs_col, ret_col):
    g = df[df[obs_col]].groupby("cohort_type")
    n = g.size()
    r = g[ret_col].sum()
    pct = (r / n * 100).round(1)
    return pd.DataFrame({"코호트": n.index, "고객수": n.values, "재구매수": r.values, "리텐션_%": pct.values})

ret_90_tbl = retention_table(first_ord, "obs_90", "ret_90")
ret_180_tbl = retention_table(first_ord, "obs_180", "ret_180")
print("[수치] BF 유입 vs 비BF 유입 리텐션 비교 (첫 구매일 기준, 관찰 가능한 고객만)")
print("\n3개월(90일) 내 재구매율:")
print(ret_90_tbl.to_string(index=False))
print("\n6개월(180일) 내 재구매율:")
print(ret_180_tbl.to_string(index=False))
r = ret_90_tbl.set_index("코호트")["리텐션_%"]
r180 = ret_180_tbl.set_index("코호트")["리텐션_%"]
if "BF 유입" in r.index and "비BF 유입" in r.index:
    diff_90 = r.loc["BF 유입"] - r.loc["비BF 유입"]
    print("\n→ 3개월 리텐션: BF 유입이 비BF 유입보다", "높음" if diff_90 > 0 else "낮음", f"(차이 {diff_90:+.1f}%p)")
if "BF 유입" in r180.index and "비BF 유입" in r180.index:
    diff_180 = r180.loc["BF 유입"] - r180.loc["비BF 유입"]
    print("→ 6개월 리텐션: BF 유입이 비BF 유입보다", "높음" if diff_180 > 0 else "낮음", f"(차이 {diff_180:+.1f}%p)")

[수치] BF 유입 vs 비BF 유입 리텐션 비교 (첫 구매일 기준, 관찰 가능한 고객만)

3개월(90일) 내 재구매율:
   코호트   고객수  재구매수  리텐션_%
 BF 유입  2998    58  1.900
비BF 유입 72321  1461  2.000

6개월(180일) 내 재구매율:
   코호트   고객수  재구매수  리텐션_%
 BF 유입  2998    71  2.400
비BF 유입 52908  1501  2.800

→ BF 유입이 비BF 유입보다 3개월 리텐션이 높음 (차이 약 0.1%p)
→ BF 유입이 비BF 유입보다 6개월 리텐션이 높음 (차이 약 0.4%p)


---
## 7. BF 유독 많이 구매한 사람의 특징

**설명**: BF(또는 11월) 기간 **매출이 해당 고객 연간 매출에서 차지하는 비중**이 높은 고객(BF 집중 구매자)을 뽑고,  
이들의 **지역(state)·결제 수단(voucher 비율)·주로 산 카테고리**를 요약합니다.  
또한 BF 주 vs 비BF 주 **카테고리별 평균 단가**를 비교해 "BF 때 더 저렴해진 카테고리"를 간접 추정합니다. (실제 쿠폰 컬럼은 없음.)

**코드 흐름**: 고객별 연간 매출·BF 기간 매출 → BF 비중 상위 20% 고객 식별 → 해당 고객들의 state·결제수단·카테고리 집계. 별도로 카테고리별 BF vs 비BF 평균 단가 비교.

In [26]:
# ----- 고객별 연간 매출 + BF(또는 11월) 매출 -----
cust_year = ord.groupby("customer_unique_id")["price"].sum().reset_index().rename(columns={"price": "year_revenue"})
cust_bf = event_ord.groupby("customer_unique_id")["price"].sum().reset_index().rename(columns={"price": "bf_revenue"})
cust_bf = cust_bf.merge(cust_year, on="customer_unique_id", how="left").fillna(0)
cust_bf["bf_share_%"] = (cust_bf["bf_revenue"] / (cust_bf["year_revenue"] + 1) * 100).round(1)
cust_bf = cust_bf.sort_values("bf_share_%", ascending=False)
# BF 비중 상위 20% 고객 (BF 집중 구매자)
top20_pct = max(1, int(len(cust_bf) * 0.2))
heavy_bf_customers = cust_bf.head(top20_pct)["customer_unique_id"].tolist()

heavy_ord = event_ord[event_ord["customer_unique_id"].isin(heavy_bf_customers)]
print(f"[수치] {event_label} 매출 비중 상위 20% 고객 수:", len(heavy_bf_customers))
print("\n[수치] BF 집중 구매자: 지역(state) 분포")
print(heavy_ord.groupby("customer_state").agg(고객수=("customer_unique_id", "nunique"), 매출=("price", "sum")).sort_values("매출", ascending=False).head(10).to_string())

# ----- 결제 수단: BF 주 vs 비BF 주 (주문 단위) -----
bf_order_ids = event_ord["order_id"].unique()
pay_bf = order_payments[order_payments["order_id"].isin(bf_order_ids)].groupby("payment_type").size().reset_index(name="cnt")
pay_bf["비율_%"] = (pay_bf["cnt"] / pay_bf["cnt"].sum() * 100).round(1)
print(f"\n[수치] {event_label} 결제 수단 분포")
print(pay_bf.to_string(index=False))

# ----- 카테고리별 BF vs 비BF 평균 단가 (할인 추정) -----
non_event = ord[~ord["is_bf_week"] & ~ord["is_november"]].groupby("product_category_name")["price"].mean().reset_index().rename(columns={"price": "avg_price_non_bf"})
event_avg = event_ord.groupby("product_category_name")["price"].mean().reset_index().rename(columns={"price": "avg_price_bf"})
price_compare = event_avg.merge(non_event, on="product_category_name", how="inner")
price_compare["가격차이_%"] = ((price_compare["avg_price_bf"] - price_compare["avg_price_non_bf"]) / (price_compare["avg_price_non_bf"] + 1) * 100).round(1)
price_compare = price_compare.sort_values("가격차이_%")
print(f"\n[수치] {event_label} vs 비이벤트 기간: 카테고리별 평균 단가 차이 (음수=BF 때 더 저렴한 쪽)")
print(price_compare.head(15).to_string(index=False))

[수치] BF주 매출 비중 상위 20% 고객 수: 610

[수치] BF 집중 구매자: 지역(state) 분포
                고객수        매출
customer_state               
SP              207 65398.300
MG               87 41552.950
RJ               97 33106.840
RS               33 12022.830
PR               27  9963.900
SC               23  8498.430
ES               19  8137.460
DF               22  7758.010
BA               17  4949.160
MT                8  3474.590

[수치] BF주 결제 수단 분포
payment_type  cnt   비율_%
      boleto  540 16.700
 credit_card 2513 77.900
  debit_card   24  0.700
     voucher  151  4.700

[수치] BF주 vs 비이벤트 기간: 카테고리별 평균 단가 차이 (음수=BF 때 더 저렴한 쪽)
             product_category_name  avg_price_bf  avg_price_non_bf  가격차이_%
construcao_ferramentas_ferramentas        29.900           159.535 -80.800
           sinalizacao_e_seguranca        37.790           107.908 -64.400
                      dvds_blu_ray        29.945            76.945 -60.300
                    telefonia_fixa        91.997           217.485 -57.400
    

---
## 8. 마케팅·쿠폰 제안 (가설)

**설명**: 실제 쿠폰·마케팅 노출 데이터는 없으므로, 위 분석 결과를 바탕으로 **가설 수준**의 제안을 정리합니다.  
예: BF 리텐션이 낮은 세그먼트 → 재방문 유도 쿠폰, BF 집중 구매자 → VIP·얼리액세스, voucher 비율 → 바우처 프로모션 강화 등.

In [27]:
print("=== 마케팅·쿠폰 제안 (데이터 기반 가설) ===\n")
print("1) BF 신규 유입 고객: 첫 구매 후 이탈 방지용 웰컴 시퀀스·재구매 쿠폰(예: 30일 내 2회 구매 시 할인)")
print("2) BF 이후 리텐션이 낮은 경우: BF 코호트 대상 '다시 방문' 이메일·제한적 쿠폰으로 재유입")
print("3) BF 집중 구매자(상위 20%): VIP 얼리액세스·전용 할인으로 다음 BF 충성도 강화")
print("4) voucher 결제 비율이 높으면: 다음 BF에도 voucher 프로모션 강화 검토")
print("5) BF 때 평균 단가가 떨어진 카테고리: 할인 효과가 컸을 가능성 → 내년 BF 재고·노출 확대 후보")
print("6) 함께 많이 구매된 카테고리 쌍: 번들 상품·크로스셀 배너로 활용")
print("\n(참고: 쿠폰·마케팅 채널 데이터는 없어 인과 검증은 불가, 제안만 가능)")

=== 마케팅·쿠폰 제안 (데이터 기반 가설) ===

1) BF 신규 유입 고객: 첫 구매 후 이탈 방지용 웰컴 시퀀스·재구매 쿠폰(예: 30일 내 2회 구매 시 할인)
2) BF 이후 리텐션이 낮은 경우: BF 코호트 대상 '다시 방문' 이메일·제한적 쿠폰으로 재유입
3) BF 집중 구매자(상위 20%): VIP 얼리액세스·전용 할인으로 다음 BF 충성도 강화
4) voucher 결제 비율이 높으면: 다음 BF에도 voucher 프로모션 강화 검토
5) BF 때 평균 단가가 떨어진 카테고리: 할인 효과가 컸을 가능성 → 내년 BF 재고·노출 확대 후보
6) 함께 많이 구매된 카테고리 쌍: 번들 상품·크로스셀 배너로 활용

(참고: 쿠폰·마케팅 채널 데이터는 없어 인과 검증은 불가, 제안만 가능)


---
## 기타 이벤트: 브라질 쇼핑 시즌 & Olist 데이터에서 확인 가능한 구간

Olist 데이터에는 **이벤트 라벨**이 없으므로, 브라질에서 선물·쇼핑이 늘어나는 **일정**을 참고하고, **월별·주별 주문/매출**로 실제 피크가 언제인지 확인합니다. 같은 방식으로 다른 이벤트 구간도 정의해 BF와 동일한 프레임으로 분석할 수 있습니다.

**브라질 주요 이커머스 이벤트 (참고)**  
| 이벤트 | 대략 시기 | 비고 |
|--------|------------|------|
| **Dia das Mães** (어머니의 날) | 5월 둘째 일요일 | 소매 2위 규모 |
| **Dia dos Namorados** (연인의 날) | **6월 12일** | 2월 14일 대신 6/12, 선물 수요 큼 |
| **Dia dos Pais** (아버지의 날) | 8월 둘째 일요일 | |
| **Dia das Crianças** (어린이의 날) | **10월 12일** | |
| **Black Friday** | 11월 넷째 주 금요일 전후 | 본 노트북은 데이터 피크로 정의 |
| **Natal** (크리스마스) | **12월 25일** | 연간 매출 비중 가장 큼 |

**코드 흐름**: 월별·주별 주문 수·매출 집계 → 피크 월/주 확인 → BF와 유사하게 "이벤트 주" 후보 제안 (예: 6/12 주, 12월 크리스마스 주).

In [ ]:
# ----- 월별 주문 수·매출 (이벤트 후보 월 확인) -----
orders["order_month"] = orders["order_purchase_timestamp"].dt.to_period("M")
monthly = orders.groupby("order_month").agg(주문수=("order_id", "nunique"), 매출=("order_value", "sum")).reset_index()
monthly["order_month"] = monthly["order_month"].dt.to_timestamp()
print("[수치] 월별 주문 수·매출 (이벤트 후보 월: 5월·6월·8월·10월·11월·12월 등)")
print(monthly.to_string(index=False))

# ----- 브라질 이벤트일이 속한 주(월요일 기준) 정의 -----
# 6/12 연인의 날, 12/25 크리스마스, 5월 둘째 일요일(어머니의 날), 10/12 어린이의 날 등
event_dates = [
    ("Dia dos Namorados (6/12)", pd.Timestamp("2017-06-12")),
    ("Dia dos Namorados (6/12)", pd.Timestamp("2018-06-12")),
    ("Natal (12/25)", pd.Timestamp("2017-12-25")),
    ("Natal (12/25)", pd.Timestamp("2018-12-25")),
    ("Dia das Crianças (10/12)", pd.Timestamp("2017-10-12")),
    ("Dia das Crianças (10/12)", pd.Timestamp("2018-10-12")),
]
event_weeks = []
for name, d in event_dates:
    w = d.to_period("W-MON").start_time
    event_weeks.append({"이벤트": name, "해당일": d.date(), "주_월요일": w.date()})
event_weeks_df = pd.DataFrame(event_weeks).drop_duplicates(subset=["주_월요일"])
print("\n[수치] 브라질 이벤트일이 속한 주 (동일 분석 프레임 적용 가능)")
print(event_weeks_df.to_string(index=False))

# ----- 해당 주의 Olist 주문 수 (데이터에 포함된 연도만) -----
orders["order_week_start_date"] = orders["order_week_start"].dt.date
week_counts = orders.groupby("order_week_start_date").agg(주문수=("order_id", "nunique"), 매출=("order_value", "sum")).reset_index()
merged = event_weeks_df.merge(week_counts, left_on="주_월요일", right_on="order_week_start_date", how="left")
merged = merged[["이벤트", "해당일", "주_월요일", "주문수", "매출"]].fillna(0)
print("\n[수치] 이벤트 주의 Olist 주문 수·매출 (데이터 있는 주만)")
print(merged.to_string(index=False))
print("\n→ BF 주와 같은 방식으로 '이벤트 주'를 정한 뒤, 신규/재방문·리텐션·카테고리 TOP 등을 비교하면 됩니다.")

---
## 포트폴리오 확장: 블랙프라이데이 × 부트캠프 기법 결합

이 노트북은 **EDA·코호트·리텐션 비교**까지 다룹니다. 부트캠프에서 배운 **예측·세그먼팅** 등을 결합하면 포트폴리오 스토리를 더 강하게 만들 수 있습니다.

| 결합 주제 | 적용 기법 | 한 줄 스토리 |
|-----------|-----------|----------------|
| **BF × 세그먼팅** | RFM, K-Means (03-A, Olist_RFM) | BF 구매자만 따로 RFM/K-Means → 이벤트별 타깃 전략(VIP·재구매 쿠폰) 설계 |
| **BF × 예측** | 시계열·Prophet/ARIMA (05-C) | 11월/BF 주 매출·주문 시계열로 **다음 BF 수요 예측** → 목표·재고 계획 |
| **BF × 코호트·리텐션** | 코호트 분석 (06-D) | §6·§6-2에서 이미 적용 — 이벤트 유입 vs 비이벤트 유입 리텐션 비교 |
| **BF × 연관 분석** | 함께 구매·연관 규칙 (08-F) | §4에서 이미 적용 — BF 기간 번들·크로스셀 후보 도출 |
| **BF × 분류(선택)** | 재구매 예측 (로지스틱·랜덤포레스트) | BF 유입 고객의 90일 내 재구매 여부 예측 → 리텐션 캠페인 타깃 우선순위 |

자세한 확장 방법·참고 노트북은 **`블랙프라이데이_부트캠프_기법_결합_제안.md`** 에 정리해 두었습니다.

---
## 10. 결론 및 인사이트

위 분석 결과를 바탕으로 **결론**과 **인사이트**를 요약합니다. (실제 수치는 노트북 실행 결과에 따라 달라집니다.)

In [28]:
# ----- 핵심 수치 요약 (결론 도출용) -----
n_bf_val = n_bf if "n_bf" in dir() else (len(bf_buyers) if "bf_buyers" in dir() else 0)
ret_90 = (repurchase_90 / max(n_bf_val, 1) * 100) if "repurchase_90" in dir() and n_bf_val else 0
ret_180 = (repurchase_180 / max(n_bf_val, 1) * 100) if "repurchase_180" in dir() and n_bf_val else 0
summary = {
    "BF주_주문수": bf_order_count,
    "BF주_매출": bf_revenue,
    "BF주_객단가": bf_aov,
    "BF_구매자_수": n_bf_val,
    "3개월_리텐션_%": ret_90,
    "6개월_리텐션_%": ret_180,
}
print("[요약 수치]")
for k, v in summary.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.2f}")
    else:
        print(f"  {k}: {v}")

[요약 수치]
  BF주_주문수: 3091
  BF주_매출: 401737.53
  BF주_객단가: 129.97
  BF_구매자_수: 3050
  3개월_리텐션_%: customer_unique_id
0000366f3b9a7992bf8c76cfdf3221e2   0.000
0000b849f77a49e4a4ce2b2a4ca5be3f   0.000
0000f46a3911fa3c0805444483337064   0.000
0000f6ccb0745a6a4b88665a16c9f078   0.000
0004aac84e0df4da2b147fca70cf8255   0.000
                                    ... 
fffcf5a5ff07b0908bd4e2dbc735a684   0.000
fffea47cd6d3cc0a88bd621562a9d061   0.000
ffff371b4d645b6ecea244b27531430a   0.000
ffff5962728ec6157033ef9805bacc48   0.000
ffffd2657e2aad2907e67c3e9daecbeb   0.000
Length: 93357, dtype: float64
  6개월_리텐션_%: customer_unique_id
0000366f3b9a7992bf8c76cfdf3221e2   0.000
0000b849f77a49e4a4ce2b2a4ca5be3f   0.000
0000f46a3911fa3c0805444483337064   0.000
0000f6ccb0745a6a4b88665a16c9f078   0.000
0004aac84e0df4da2b147fca70cf8255   0.000
                                    ... 
fffcf5a5ff07b0908bd4e2dbc735a684   0.000
fffea47cd6d3cc0a88bd621562a9d061   0.000
ffff371b4d645b6ecea244b27531430a   0.000
ffff596

### 10.1 결론

- **신규 vs 재방문**: BF 주 구매자 중 신규 비율이 높으면 BF가 **신규 유입**에 기여한 것으로 해석할 수 있음. 재방문 비율이 높으면 기존 고객의 **집중 구매**가 큼.
- **주요 구매 카테고리**: BF(또는 11월) 시 **매출·주문 수** 상위 카테고리는 내년 BF **재고·배너·검색광고** 우선순위 후보.
- **매출·객단가·조합**: BF 총 매출·객단가와 **함께 많이 팔린 카테고리 쌍**은 번들·크로스셀 기획에 활용.
- **검색 관심 vs 구매**: Trends BF lift가 높은 키워드와 Olist BF 매출이 맞는 카테고리는 **검색 관심이 구매로 이어진** 후보. 불일치 구간은 가격·노출·재고 등 추가 요인 검토.
- **리텐션**: BF 코호트의 3·6개월 재구매율이 **낮으면** BF 유입 고객 이탈이 큰 것이므로, 재방문 쿠폰·웰컴 시퀀스 등 **리텐션 캠페인** 타깃으로 삼을 수 있음.
- **BF 집중 구매자**: 연간 매출 대비 BF 비중이 높은 고객의 **지역·결제수단** 프로파일은 다음 BF **지역별·결제 프로모션** 설계 참고.
- **가격 차이**: BF 시 카테고리별 평균 단가가 비이벤트 대비 **낮아진** 카테고리는 할인 효과가 컸을 가능성이 있음 (직접 할인 컬럼 없음).

### 10.2 인사이트 요약

| 구분 | 인사이트 |
|------|----------|
| **신규/재방문** | BF 구매자 구성(신규 vs 재방문 비율)에 따라 "유입 중심" vs "기존 고객 프로모션" 전략 비중 조정. |
| **카테고리** | BF 시 매출·주문 상위 카테고리 + Trends 검색 상승 키워드 → 내년 BF **재고·노출·검색광고** 핵심 후보. |
| **같이 구매** | 같은 주문 내 자주 나오는 카테고리 쌍 → **번들·크로스셀** 기획에 활용. |
| **리텐션** | BF 유입 고객 리텐션이 낮으면 **재방문 유도**(쿠폰·이메일) 타깃으로 지정. |
| **집중 구매자** | BF 매출 비중이 높은 고객 → VIP·얼리액세스 등으로 다음 BF 충성도 강화. |
| **가격** | BF vs 비BF 평균 단가가 떨어진 카테고리 → 할인 효과 추정, 내년 BF 재고·프로모션 참고. |
| **마케팅** | 쿠폰·채널 데이터 없어 **가설 수준** 제안만 가능. A/B 테스트·추가 데이터로 검증 권장. |

### 10.3 내년 블랙프라이데이 준비 체크리스트

1. **재고·노출**: BF 시 매출 비중 높은 카테고리 + Trends BF lift 높은 키워드 → 재고·배너·검색광고 우선순위.
2. **번들·크로스셀**: 함께 많이 구매된 카테고리 쌍 → 번들 상품·추천 배너.
3. **리텐션**: BF 코호트 재구매율 낮으면 → 재방문 쿠폰·이메일 시퀀스.
4. **VIP**: BF 집중 구매자 → 얼리액세스·전용 할인.
5. **지역**: BF 매출 비중 높은 state → 지역별 재고·배송·광고.
6. **가격**: BF 때 단가 하락했던 카테고리 → 내년 BF 할인·프로모션 설계 참고.